# holdout-data-one-per-class — worked example 2: Holdout gallery handles imbalanced class frequencies

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `holdout-data-one-per-class`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Real datasets are often class-imbalanced: some classes have many samples, others few. The one-per-class pattern is robust to imbalance because `labels == c` correctly finds all samples of any class regardless of how many there are, as long as at least one exists. The result is always a balanced gallery of exactly one sample per class, regardless of the original distribution.

## Worked solution

**Step 1 — build an imbalanced synthetic dataset.** We create 3 classes with very different frequencies: class 0 has 40 samples, class 1 has 5, class 2 has 1. The function must still find each class.

**Step 2 — run one_per_class.** Call the function. With only one sample in class 2, the mask selects exactly one item; `data[mask][0]` still works.

**Step 3 — verify output shape.** The gallery should be `(3, feature_dim)` regardless of the original distribution.

**Step 4 — confirm the rare-class sample was selected.** For class 2 (only one sample in the entire dataset), verify the holdout equals that one sample exactly.

In [ ]:
import torch as t

t.manual_seed(20)

def one_per_class(data: t.Tensor, labels: t.Tensor, num_classes: int) -> t.Tensor:
    per_class = []
    for c in range(num_classes):
        mask = labels == c
        per_class.append(data[mask][0])
    return t.stack(per_class, dim=0)

# Imbalanced dataset: class 0 has 40, class 1 has 5, class 2 has 1 sample
t.manual_seed(20)
D = 6
data_parts = [
    t.randn(40, D),  # class 0
    t.randn(5, D),   # class 1
    t.randn(1, D),   # class 2 — only one!
]
data = t.cat(data_parts, dim=0)         # (46, 6)
labels = t.cat([
    t.zeros(40, dtype=t.long),
    t.ones(5, dtype=t.long),
    t.full((1,), 2, dtype=t.long),
])                                       # (46,)

# Shuffle
perm = t.randperm(46, generator=t.Generator().manual_seed(99))
data, labels = data[perm], labels[perm]

holdout = one_per_class(data, labels, 3)
print(f"Holdout shape: {holdout.shape}")  # (3, 6)
assert holdout.shape == (3, D)

# The class-2 holdout must equal the one and only class-2 sample
only_class2 = data[labels == 2][0]
assert t.allclose(holdout[2], only_class2), "Rare-class sample not selected correctly"
print("Imbalanced-class test passed: all 3 classes present in holdout.")